# 02 — SHAP Deep Dive

SHAP (SHapley Additive exPlanations) connects game-theoretic fairness to feature attribution.

## Shapley Value Derivation

The Shapley value for feature *i* is the average marginal contribution of *i* across all possible orderings of features:

$$\phi_i = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!(|F|-|S|-1)!}{|F|!} \left[ v(S \cup \{i\}) - v(S) \right]$$

where *v(S)* is the model's prediction given only the features in subset *S* (others marginalised out). This satisfies four axioms:
- **Efficiency**: attributions sum to the prediction minus the base value
- **Symmetry**: equally contributing features get equal attribution
- **Dummy**: features that never change the output get zero attribution
- **Linearity**: attributions compose across additive models

## TreeSHAP Internals

Naive Shapley computation is exponential in the number of features. TreeSHAP (Lundberg et al., 2018) exploits the tree structure to compute exact Shapley values in O(TLD²) where T is the number of trees, L is the number of leaves, and D is the tree depth. It tracks the fraction of training samples that pass through each branch, using them as the conditional distribution for marginalisation — avoiding the 'observational vs interventional' ambiguity that plagues kernel-based approaches.

In [ ]:
# SHAP: TreeExplainer implementation from scratch
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5,
                            n_redundant=2, random_state=42)
feature_names = [f'f{i}' for i in range(10)]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

clf = GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42)
clf.fit(X_train, y_train)
print(f'Test accuracy: {clf.score(X_test, y_test):.3f}')

In [ ]:
# Kernel SHAP approximation (model-agnostic)
def kernel_shap(model, x, X_background, n_samples=512):
    """Approximate Shapley values via weighted linear regression."""
    n_features = x.shape[0]
    background_mean = X_background.mean(axis=0)
    base_val = model.predict_proba(background_mean.reshape(1, -1))[0, 1]

    # Sample random subsets
    masks = np.random.randint(0, 2, size=(n_samples, n_features))
    # Ensure all-ones and all-zeros are included
    masks[0] = 1
    masks[1] = 0

    # Build masked samples
    X_masked = np.where(masks, x, background_mean)
    preds = model.predict_proba(X_masked)[:, 1]

    # Shapley kernel weights
    z = masks.sum(axis=1)
    z = np.clip(z, 1, n_features - 1)
    from math import comb
    weights = np.array([
        (n_features - 1) / (comb(n_features, int(zi)) * int(zi) * (n_features - int(zi)))
        for zi in z
    ])

    # Weighted regression
    from numpy.linalg import lstsq
    W = np.diag(weights)
    phi, _, _, _ = lstsq(masks.T @ W @ masks, masks.T @ W @ (preds - base_val),
                         rcond=None)
    return phi, base_val

x_sample = X_test[0]
shap_vals, base = kernel_shap(clf, x_sample, X_train, n_samples=1024)
print(f'Base value: {base:.4f}')
print(f'Prediction: {clf.predict_proba(x_sample.reshape(1,-1))[0,1]:.4f}')
print(f'Sum of SHAPs + base: {shap_vals.sum() + base:.4f}')
for i, v in enumerate(shap_vals):
    print(f'  {feature_names[i]}: {v:+.4f}')

In [ ]:
# Visualise SHAP values
import matplotlib.pyplot as plt

sorted_idx = np.argsort(np.abs(shap_vals))[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['steelblue' if v > 0 else 'tomato' for v in shap_vals[sorted_idx]]
ax.barh([feature_names[i] for i in sorted_idx], shap_vals[sorted_idx], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('SHAP value (impact on model output)')
ax.set_title('SHAP Local Explanation — Sample 0')
plt.tight_layout()
plt.savefig('/tmp/shap_local.png', dpi=100, bbox_inches='tight')
plt.show()
print('SHAP local explanation saved')

In [ ]:
# Linear SHAP — exact for linear models
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

def linear_shap(model, x, X_background):
    """For linear models, SHAP = coef * (x - E[x])."""
    # For log-odds output
    background_mean = X_background.mean(axis=0)
    phi = model.coef_[0] * (x - background_mean)
    base = model.predict_proba(background_mean.reshape(1,-1))[0,1]
    return phi, base

lin_shap, lin_base = linear_shap(lr, X_test[0], X_train)
print('Linear SHAP values:')
for i, v in enumerate(lin_shap):
    print(f'  {feature_names[i]}: {v:+.4f}')
print(f'\nEfficiency check — base + sum ≈ prediction:')
print(f'  Base: {lin_base:.4f}')
print(f'  Sum SHAPs: {lin_shap.sum():.4f}')
print(f'  LR prediction: {lr.predict_proba(X_test[0].reshape(1,-1))[0,1]:.4f}')